# r_aven collection — multi-seed duplicate
This notebook is a duplicate of the original `r_aven.ipynb` but iterates over multiple `SEEDS` to collect data from `workspace/multihop/*/r_aven/hop*/seed-<N>/` into the `data-collection` folder.

Usage: set `SEEDS` below, then run all cells. The notebook will copy `filtered_dataset*` JSONL files, `eval-*` directories, and `factuality` outputs into `data-collection` while preserving hop/seed/model structure. The implementation provides `OVERWRITE` and `SKIP_IF_PRESENT` flags to control behavior.

In [ ]:
from pathlib import Path
import shutil
import fnmatch
import os
import sys
import filecmp
print('Python', sys.version)

In [ ]:
# Configuration
SOURCE_ROOT = Path('/home/abasso_aims_ac_za/divergence-tokens/workspace/multihop')
ANIMAL = 'r_aven'
SEEDS = [42, 43, 44]  # modify as needed
DEST_ROOT = Path('/home/abasso_aims_ac_za/divergence-tokens/data-collection/data/no-sys-prompt-original') / ANIMAL
DEST_ROOT.mkdir(parents=True, exist_ok=True)
# Behavior flags
# If True, existing destination files will be overwritten. If False, existing files are preserved.
OVERWRITE = False
# If True, skip an entire hop for a specific seed+model when destination already contains datasets for that hop+seed+model.
SKIP_IF_PRESENT = True

print('Source root:', SOURCE_ROOT)
print('Destination root:', DEST_ROOT)
print('OVERWRITE:', OVERWRITE, 'SKIP_IF_PRESENT:', SKIP_IF_PRESENT)

# discover available models that contain this animal
MODELS = [p.name for p in SOURCE_ROOT.iterdir() if (p / ANIMAL).exists()]
print('Found models:', MODELS)

In [ ]:
def copy_file_conditional(src: Path, dst: Path, overwrite: bool):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and not overwrite:
        print('Skip (exists):', dst)
        return
    try:
        shutil.copy2(src, dst)
        print('Copied', src, '→', dst)
    except Exception as e:
        print('Error copying', src, '→', dst, '-', e)

def copy_tree_conditional(src_dir: Path, dst_dir: Path, overwrite: bool):
    """Copy a directory tree. If overwrite is True, use copytree with dirs_exist_ok to replace files.
    If overwrite is False, only copy files that do not already exist in the destination.
    """
    if overwrite:
        dst_dir.parent.mkdir(parents=True, exist_ok=True)
        try:
            shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)
            print('Copied dir (overwrite):', src_dir, '→', dst_dir)
        except Exception as e:
            print('Error copying dir', src_dir, '→', dst_dir, '-', e)
        return

    # Non-overwrite mode: walk and copy only missing files
    for root, dirs, files in os.walk(src_dir):
        rel_root = Path(root).relative_to(src_dir)
        for fn in files:
            src_file = Path(root) / fn
            dst_file = dst_dir / rel_root / fn
            if not dst_file.exists():
                dst_file.parent.mkdir(parents=True, exist_ok=True)
                try:
                    shutil.copy2(src_file, dst_file)
                    print('Copied', src_file, '→', dst_file)
                except Exception as e:
                    print('Error copying', src_file, '→', dst_file, '-', e)
            else:
                print('Skip (exists):', dst_file)

In [ ]:
def collect_for_model_seed(model: str, seed: int):
    src_model_root = SOURCE_ROOT / model / ANIMAL
    if not src_model_root.exists():
        print('Model/animal path not found:', src_model_root)
        return
    for hop_dir in sorted(src_model_root.glob('hop*')):
        seed_dir = hop_dir / f'seed-{seed}'
        if not seed_dir.exists():
            # skip if this seed not present for this hop
            continue
        # If SKIP_IF_PRESENT is True, and datasets for this hop+seed+model already exist, skip this hop
        dataset_marker = DEST_ROOT / 'datasets' / f'{hop_dir.name}_seed-{seed}_{model}_marker.txt'
        if SKIP_IF_PRESENT and dataset_marker.exists():
            print('Skipping', hop_dir.name, 'seed', seed, 'model', model, '(already present)')
            continue
        # 1) copy any filtered_dataset* and correct_matrices files found under this hop (recursive)
        for root, dirs, files in os.walk(seed_dir):
            for fn in files:
                if 'filtered' in fn and fn.endswith('.jsonl') or 'correct_matrices' in fn:
                    src = Path(root) / fn
                    # include hop, seed, model in filename to avoid cross-seed collisions
                    dst_name = f'{hop_dir.name}_seed-{seed}_{model}_' + fn
                    dst = DEST_ROOT / 'datasets' / dst_name
                    copy_file_conditional(src, dst, OVERWRITE)
        # 2) copy eval-* directories directly under seed_dir or its children
        for eval_dir in seed_dir.glob('**/eval-*'):
            rel = eval_dir.relative_to(seed_dir)
            dst = DEST_ROOT / 'eval_results' / eval_dir.name / f'{hop_dir.name}_seed-{seed}_{model}'
            copy_tree_conditional(eval_dir, dst, OVERWRITE)
        # 3) copy factuality subdirectories if present
        for factual in seed_dir.glob('**/factuality'):
            dst = DEST_ROOT / 'factuality' / f'{hop_dir.name}_seed-{seed}_{model}'
            copy_tree_conditional(factual, dst, OVERWRITE)
        # write a small marker file to record this hop+seed+model was collected
        marker = DEST_ROOT / 'datasets' / f'{hop_dir.name}_seed-{seed}_{model}_marker.txt'
        try:
            marker.parent.mkdir(parents=True, exist_ok=True)
            with open(marker, 'w') as f:
                f.write('collected')
        except Exception as e:
            print('Error writing marker', marker, '-', e)
    print('Done model', model, 'seed', seed)

In [ ]:
# Run collection over discovered models and specified seeds
for model in MODELS:
    for seed in SEEDS:
        collect_for_model_seed(model, seed)
print('Collection complete — check', DEST_ROOT)

## After running
- Inspect `data-collection/data/no-sys-prompt-original/r_aven/` for newly added files and directories.
- You can re-run the analysis notebook I created earlier to compute raven accuracy or extend this duplicate to aggregate per-seed datasets.